In [122]:
import sys
sys.path.append("../src/")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [123]:
from ipm_newer.ipm_solve_no_split import ipm_solve_no_split # type: ignore[reportMissingImports]
from relaxers.remap_lower_bounds import remap_lower_bounds # type: ignore[reportMissingImports]
from relaxers.quadratic_model_lin_approx import quadratic_model_lin_approx # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp_return_x # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np

In [124]:
LIN_APPROX = 10

# Whether to do an inner or outer approximation
OUTER_APPROXIMATION = False

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

f_name = "model11_quad_reform.json"

In [125]:
quad, q_time, q_x = solve_qcp_return_x(f_name)
print(quad)

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 7 8845HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Presolve a model with 0 rows, 20 columns and 0 nonzeros (Min)
Model fingerprint: 0xdd0b2740
Model has 10 linear objective coefficients
Model has 20 quadratic constraints
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  QMatrix range    [2e-02, 2e-02]
  QLMatrix range   [1e+00, 2e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [5e+01, 5e+03]
  RHS range        [0e+00, 0e+00]
  QRHS range       [8e+01, 4e+03]
Presolve removed 0 rows and 6 columns
Statistics for model 'qcp_from_json_pre':
  Problem type                : QCP
  Linear constraint matrix    : 22 rows, 22 columns, 44 nonzeros
  Variable types              : 22 continuous, 0 integer (0 binary)
  Quadratic constraints       : 4 constrs, 12 nonzeros
  Matri

In [126]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                            f_name=f_name, endpoints=False)
remap_lower_bounds(f_name="linear_approx.json")
val, time, x, compl = ipm_solve_no_split()
print(x)

print(condition_number_nes_basic())
print(f"Error: {val - quad}")

Solution is below target precision.
The algorithm stopped after 51 iterations in 0.05 seconds.

Primal objective:        -4.77817382e+03
Dual objective:          -4.77817382e+03

Norm of Primal residual: 1.94e-15
Norm of Dual residual:   5.85e-12
Abs. Compl. Gap:         2.23e-09
Rel. Compl. Gap:         2.34e-13

[ 110.          153.7037037   140.          100.           53.7037037   146.2962963   120.           80.          160.          200.         1247.09322644  644.72094808  692.91073035 2122.88394478
 2640.         1760.3144076   242.4014217   391.72459117 1739.77517575 2000.        ]
4141.1601186349235
Error: 86.02808998448381


In [ ]:
# Define a set of functions for each

quads = [[val] for val in x[:10]]
# print(quads)
iters = 5

for i in range(1, iters + 1):
    print(f"Iteration {i}")
    points_functions = []

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
            refinement_points = len(qs)
            uniform = np.linspace(lower, upper, num - refinement_points)
            for q in qs:
                idx = np.searchsorted(uniform, q)
                uniform = np.insert(uniform, idx, q)
            return uniform
        return np_uniform_add_qs

    for qs in quads:
        points_functions.append(np_uniform_add_q_factory(qs))

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + i, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name=f_name,
                                                endpoints=False)
    remap_lower_bounds(f_name="linear_approx.json")

    val, time, x, compl = ipm_solve_no_split(f_name="linear_approx.json")

    # print(f"x: {x}")
    print(f"val: {val}")
    print(f"Error: {val - quad}")

    for j, qs in enumerate(quads):
        qs.append(x[j])

    print(quads)
    # print(compl)

# print(x)
# print(f"Error: {val - quad}")

Iteration 1
Solution is below target precision.
The algorithm stopped after 51 iterations in 0.06 seconds.

Primal objective:        -4.69692506e+03
Dual objective:          -4.69692506e+03

Norm of Primal residual: 2.15e-15
Norm of Dual residual:   5.49e-12
Abs. Compl. Gap:         5.09e-09
Rel. Compl. Gap:         5.42e-13

val: 4696.925061094802
Error: 4.779328718034776
[[np.float64(110.00000000000001), np.float64(110.0)], [np.float64(153.70370370370054), np.float64(146.61525278491416)], [np.float64(140.0), np.float64(140.0)], [np.float64(100.0), np.float64(100.0)], [np.float64(53.70370370370055), np.float64(46.61525278491417)], [np.float64(146.29629629629946), np.float64(153.38474721508584)], [np.float64(120.0), np.float64(120.0)], [np.float64(80.0), np.float64(80.0)], [np.float64(160.0), np.float64(160.0)], [np.float64(200.0), np.float64(200.0)]]
Iteration 2
Solution is below target precision.
The algorithm stopped after 53 iterations in 0.05 seconds.

Primal objective:        -4.